<img src=https://courseware.decoded.com/images/decoded/logo-decoded.png align=left width=100px/>

# M07 Rapid Machine Learning

---
<small>© 2026 Decoded Limited. All rights reserved. Website: https://decoded.com </small>

### Rapid Machine Learning with PyCaret
Welcome! In this notebook, we will use **PyCaret** to rapidly build, tune, and deploy machine learning models. 

We are taking it a step further than basic scripts: we will build a **reusable Python module** capable of handling *both* **Classification** and **Regression** tasks, use **scikit-learn** for robust data splitting, and then save (pickle) our pipelines for future use.

**PyCaret** is designed to make this incredibly easy. In this notebook, we will walk through the core PyCaret workflow:
1. **Setup:** Prepare the data.
2. **Compare:** Rapidly test multiple algorithms to find the best one.
3. **Tune:** Optimize the winning algorithm's parameters.
4. **Deploy:** Save the final model for real-world use.

In [ ]:
# Install pycaret (uncomment the line below if running for the first time in this environment)
# !pip install pycaret

# 0.Loading The Data From A GitHub Repository

Follow the steps below on your terminal 

Open a new terminal:

use ls to see the files in the current directory

```bash
lab00@hub3050:~$ ls
```

then use cd to navigate to the directory we want the data to be in 

```bash
lab00@hub3050:~$ cd Documents
```

for this tutorial we want the data to be in a directory called my-work, so we will navigate to that directory

```bash
lab00@hub3050:~$ cd my-work
```

now we will use git clone to clone the repository that contains the data for this tutorial

```bash
lab00@hub3050:~/Documents/my-work$ git clone https://github.com/bostonpaul-rgb/product_folder.git
```

you should see a output similar to this:

```bash
Cloning into 'product_folder'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), 13.34 KiB | 13.34 MiB/s, done.
```

now we can navigate to the product_folder directory to see the data 



## 1. The Data: Loading and Splitting
We need two datasets to experiment workflows studied in this accelerator programme:
1. **Juice (Classification):** Predicting customer beverage brand preferences.
2. **Insurance (Regression):** Predicting medical charges based on patient attributes.

We will use `scikit-learn` to cleanly split off 20% of our data as an "unseen" holdout set to test our final loaded pipelines.

In [ ]:
from pycaret.datasets import get_data
from sklearn.model_selection import train_test_split
import pandas as pd


# 2. Regression Data
reg_data = get_data('insurance')




In [ ]:
reg_data.info()

## 2 Building the rapid model build pipeline

#### Step 1: Setup the Environment
The `setup()` function initializes the training environment and creates the transformation pipeline. It automatically infers data types and prepares everything for modeling behind the scenes.

In [ ]:
from pycaret.regression import *

# We tell PyCaret what dataset to use and which column is our target.
# session_id ensures our results are reproducible (like a random seed).
# Set charges as target variable and use 80% of data for training.
exp_setup = setup(data=reg_data, target='charges', session_id=123 , fold=5, n_jobs=1, train_size=0.8)


#### Step 2: Rapid Model Build (`compare_models`)
Instead of guessing which algorithm will work best, we use `compare_models()`. This function trains and evaluates a library of over 15 standard machine learning algorithms using cross-validation, and ranks them by performance.

In [ ]:
# This will output a leaderboard of models. 
# It returns the top-performing model object.

# You can add arguments to res
best_model = compare_models(fold = 5, sort = 'RMSE', n_select = 1, budget_time=0.3)


#### Step 3: Inspecting Model Parameters
What exactly is `best_model`? It is an actual scikit-learn (or similar) model object with default parameters. Let's print it to see its internal settings.

In [ ]:
print("--- Best Baseline Model Parameters ---")
print(best_model)

#### Step 4: Tune the Model
Our `best_model` did well, but it is just using default settings. The `tune_model()` function automatically searches through a grid of hyperparameters to squeeze out even better performance.

In [ ]:
# Tune the hyperparameters of the best model
tuned_model = tune_model(best_model, n_iter=5, return_train_score=True)

# Let's look at the parameters now. Notice how they have changed from the baseline!
print("\n--- Tuned Model Parameters ---")
print(tuned_model)



#### Step 5: Deploy (Save the Model)
Once we are happy with our tuned model, we need to save it so we can use it in a real application (like a web app or an automated script). 

`save_model()` bundles the tuned model AND the entire data preparation pipeline into a single downloadable file.

In [ ]:
# Save the model to your hard drive
# Save the model to your hard drive
save_model(tuned_model, 'my_first_regression_pipeline')

print("\nSuccess! Your model is now saved as 'my_first_regression_pipeline.pkl' and ready for the real world.")

### 3. Model Pickling & Inference on test data
Finally, we save both pipelines, load them back up using the respective PyCaret function, and make predictions on the unseen data we set aside earlier.

In [ ]:
# 1. Load the saved pipeline
# Notice we don't need to add the '.pkl' extension, PyCaret handles it!
print("Loading saved pipeline...")
loaded_pipeline = load_model('my_first_regression_pipeline')

In [ ]:
# 2. Make predictions on the test/holdout data
# By default, if we don't pass a new dataframe to the 'data' parameter, 
# PyCaret automatically uses the test set it created during setup().
print("\nGenerating predictions on the test data...")
predictions = predict_model(loaded_pipeline, data=reg_data)


In [ ]:
# 3. View the results
# PyCaret adds two new columns to the end of the dataframe:
display(predictions.head(10))

### 4 Visualising Model Performance 
Numbers are great, but visuals are better. We want to see exactly how many predictions our model got right, and where it got "confused" (e.g., predicting Citrus Hill when the customer actually bought Minute Maid).

PyCaret's `plot_model()` function automatically evaluates your trained model on the holdout test set and generates beautiful interactive charts. 

In [ ]:
# This is a regression model so the 'Label' column contains the predicted values of the target variable (charges in this case).

# Try getting feature importance for the tuned model. This will show you which features had the biggest impact on the model's predictions.


# Note: You can also explore other plots just by changing the 'plot' parameter!
# Try uncommenting the lines below:
plot_model(tuned_model, plot='feature')  # Feature Importance
# plot_model(tuned_model, plot='auc')      # ROC-AUC Curve

## 5 Build an app for making predictions with unseen data

In [ ]:
# Now build your app for regression dta

# 1. Load your model
from pycaret.regression import *

# 1. Load your model
lr = load_model('my_first_regression_pipeline')

# 2. Initialise the environment 
# You must pass the dataset (or a sample) used to train the model


# 3. Now create the app




In [ ]:
# 2. Initialise the environment 
# You must pass the dataset (or a sample) used to train the model
setup(data=reg_data, target='charges', session_id = 123)

In [ ]:
# 3. Now create the app
create_app(lr)

